<h1>RAZ Systems </h1>

# Assignment -- AutoGen Core: Joke Pipeline (external chaining, no LLM agent)

### Problem Statement

This is a **different pattern** from the previous duel/judge assignments -- there, one orchestrator agent called sub-agents *from inside its own handler*. Here, **you** do the chaining, from your own calling code: send a message to one agent, take its reply, feed that reply as the input to a *second*, completely different kind of agent -- one that **does not call an LLM at all**.

You'll build:
- `ComedianAgent` -- a `RoutedAgent` that delegates to an `AssistantAgent` (LLM) and writes a short joke about a given topic
- `WordCounterAgent` -- a `RoutedAgent` with **no model client, no AssistantAgent, no LLM call whatsoever** -- just plain Python string logic that counts words

This mirrors the lesson's `LLMAgent -> SimpleAgent -> LLMAgent` demo, where the second agent (`SimpleAgent`) was pure Python -- a useful reminder that `RoutedAgent` is a general-purpose message handler, not an LLM-only concept.

**Your task:** fill in every `# TODO`. A full solution is at the end -- try not to peek until you've had a go.

In [ ]:
# --- Imports ---
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)

### First concept: Define the Message object

**TODO:** same as always -- a dataclass with a single `content: str` field.

In [ ]:
# --- TODO ---
@dataclass
class Message:
    ___: ___   # TODO: content: str

### Second concept: ComedianAgent -- a RoutedAgent that delegates to an LLM

Same delegation pattern as the lesson's `MyLLMAgent`.

**TODO:** build `ComedianAgent`, with a `system_message` describing a witty comedian, and a `@message_handler` that wraps the incoming message in a `TextMessage`, runs it through the delegate, and returns the joke as a new `Message`.

In [ ]:
# --- TODO ---
class ComedianAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("ComedianAgent")
        model_client = ___  # TODO: OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            "ComedianAgent",
            model_client=model_client,
            system_message=___,  # TODO: a short, witty, family-friendly comedian persona
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received topic: {message.content}")
        text_message = ___  # TODO: TextMessage(content=message.content, source="user")
        response = await ___  # TODO: self._delegate.on_messages([text_message], ctx.cancellation_token)
        joke = ___  # TODO: response.chat_message.content
        print(f"{self.id.type} wrote: {joke}")
        return Message(content=joke)

### Third concept: WordCounterAgent -- a RoutedAgent with NO LLM at all

This is the genuinely new idea here: `RoutedAgent` is just a base class for *anything* that reacts to messages -- it has no requirement to ever touch a model client. This agent's `__init__` doesn't create a model client or an `AssistantAgent` at all -- its `@message_handler` is just plain Python.

**TODO:** write a handler that counts the words in `message.content` (hint: `len(message.content.split())`) and returns a new `Message` reporting the count alongside the original text.

In [ ]:
# --- TODO ---
class WordCounterAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("WordCounterAgent")
        # Notice: no model_client, no AssistantAgent here at all.

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received: {message.content}")
        word_count = ___  # TODO: count the words in message.content
        report = ___  # TODO: build a string like f"({word_count} words) {message.content}"
        return Message(content=report)

### Fourth concept: Register both agents and start the runtime

**TODO:** register `ComedianAgent` as `"comedian"` and `WordCounterAgent` as `"word_counter"`.

In [ ]:
# --- TODO ---
runtime = SingleThreadedAgentRuntime()
await ComedianAgent.register(runtime, ___, lambda: ComedianAgent())     # TODO: "comedian"
await WordCounterAgent.register(runtime, ___, lambda: WordCounterAgent())  # TODO: "word_counter"
runtime.start()

### Fifth concept: Chain them yourself -- this is the new pattern

Unlike the duel/judge assignments, **no single agent calls another internally here**. Instead, *your own code* sends a message to the Comedian, takes the reply, and sends *that* as the input to the Word Counter. The chaining lives in your calling code, not inside any one agent's handler.

**TODO:** send a topic to the Comedian, then feed its joke into the Word Counter, then print both results.

In [ ]:
# --- TODO ---
comedian_id = ___  # TODO: AgentId("comedian", "default")
counter_id = ___   # TODO: AgentId("word_counter", "default")

joke_response = await runtime.send_message(Message(content="cats"), ___)  # TODO: comedian_id
print(">>> Joke:", joke_response.content)

count_response = await runtime.send_message(Message(content=joke_response.content), ___)  # TODO: counter_id
print(">>> Word count report:", count_response.content)

**TODO:** stop and close the runtime.

In [ ]:
# --- TODO ---
await ___  # TODO: runtime.stop()
await ___  # TODO: runtime.close()

---
## What to try next

- Chain a third step: send the word-count report back to the Comedian and ask it to shorten the joke if the count is too high
- Add a second non-LLM agent, e.g. one that checks whether the joke contains a question mark (a common setup/punchline structure) -- no model client needed for that either
- Try swapping the chain order: word-count first on the raw topic, then send to the Comedian -- notice the pipeline only works in the order you actually wire it, nothing enforces a "correct" sequence for you

---
# Solution

No peeking until you've tried it yourself!

In [2]:
# === Setup ===
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)

@dataclass
class Message:
    content: str

In [4]:
# === ComedianAgent -- delegates to an LLM ===
class ComedianAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("ComedianAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            "ComedianAgent",
            model_client=model_client,
            system_message="You are a witty, family-friendly comedian. Write one short joke, two sentences max.",
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received topic: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        joke = response.chat_message.content
        print(f"{self.id.type} wrote: {joke}")
        return Message(content=joke)

In [5]:
# === WordCounterAgent -- pure Python, NO LLM at all ===
class WordCounterAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("WordCounterAgent")
        # Notice: no model_client, no AssistantAgent here at all.

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received: {message.content}")
        word_count = len(message.content.split())
        report = f"({word_count} words) {message.content}"
        return Message(content=report)

In [ ]:
# === Register and start ===
runtime = SingleThreadedAgentRuntime()
await ComedianAgent.register(runtime, "comedian", lambda: ComedianAgent())
await WordCounterAgent.register(runtime, "word_counter", lambda: WordCounterAgent())
runtime.start()

comedian received topic: cats
comedian wrote: Why don’t cats play poker in the jungle? Because there are too many cheetahs!
word_counter received: Why don’t cats play poker in the jungle? Because there are too many cheetahs!


In [7]:
# === Chain them from our own calling code ===
comedian_id = AgentId("comedian", "default")
counter_id = AgentId("word_counter", "default")

joke_response = await runtime.send_message(Message(content="cats"), comedian_id)
print(">>> Joke:", joke_response.content)

count_response = await runtime.send_message(Message(content=joke_response.content), counter_id)
print(">>> Word count report:", count_response.content)

>>> Joke: Why don’t cats play poker in the jungle? Because there are too many cheetahs!
>>> Word count report: (14 words) Why don’t cats play poker in the jungle? Because there are too many cheetahs!


In [8]:
# === Shut down ===
await runtime.stop()
await runtime.close()